In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

base_path = "/Volumes/workspace/apex_retail/raw_landing_zone"
catalog = "workspace"
schema = "apex_retail"

# ---------- Load Bronze ----------
df_customer = spark.table(f"{catalog}.{schema}.bronze_customer")
df_product  = spark.table(f"{catalog}.{schema}.bronze_product")
df_sales    = spark.table(f"{catalog}.{schema}.bronze_sales")

# ---------- Cleaning functions ----------
def clean_dataset(df, pk_col):
    df = df.filter(F.col(pk_col).isNotNull() & (F.trim(F.col(pk_col)) != ""))
    w = Window.partitionBy(pk_col).orderBy(F.monotonically_increasing_id())
    df = df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
    return df

def cast_and_fill(df, dataset_name):
    if dataset_name == "customer":
        df = (df.withColumn("age", F.col("age").cast("int"))
                .withColumn("membership_years", F.col("membership_years").cast("int"))
                .withColumn("number_of_children", F.col("number_of_children").cast("int")))
    elif dataset_name == "product":
        df = (df.withColumn("unit_price", F.col("unit_price").cast("double"))
                .withColumn("product_rating", F.col("product_rating").cast("double"))
                .withColumn("product_review_count", F.col("product_review_count").cast("int"))
                .withColumn("product_stock", F.col("product_stock").cast("int"))
                .withColumn("product_return_rate", F.col("product_return_rate").cast("double")))
    elif dataset_name == "sales":
        df = (df.withColumn("quantity", F.col("quantity").cast("int"))
                .withColumn("unit_price", F.col("unit_price").cast("double"))
                .withColumn("discount_applied", F.col("discount_applied").cast("double"))
                .withColumn("total_sales", F.col("total_sales").cast("double")))
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    numeric_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() != "string"]
    df = df.fillna("Unknown", subset=string_cols)
    df = df.fillna(0.0, subset=[c for c in numeric_cols if c != "ingested_at"])
    return df

def split_clean(df, pk_col, dataset_name):
    hist_raw = df.filter("source_load = 'historical'")
    incr_raw = df.filter("source_load = 'incremental'")
    hist_clean = cast_and_fill(clean_dataset(hist_raw, pk_col), dataset_name)
    incr_clean = cast_and_fill(clean_dataset(incr_raw, pk_col), dataset_name)
    print(f"{dataset_name} historical: {hist_raw.count()} raw -> {hist_clean.count()} cleaned")
    print(f"{dataset_name} incremental: {incr_raw.count()} raw -> {incr_clean.count()} cleaned")
    return hist_clean, incr_clean

customer_hist_clean, customer_incr_clean = split_clean(df_customer, "customer_id", "customer")
product_hist_clean, product_incr_clean = split_clean(df_product, "product_id", "product")
sales_hist_clean, sales_incr_clean = split_clean(df_sales, "transaction_id", "sales")

# ---------- Silver audit checkpoints ----------
audit_silver_hist_files = {"customer": "customer_silver_audit.csv", "product": "product_silver_audit.csv", "sales": "sales_silver_audit.csv"}
audit_silver_new_files = {"customer": "customer_incrementalaudit_silver.csv", "product": "product_incrementalaudit_silver.csv", "sales": "sales_incrementalaudit_silver.csv"}
hist_clean_dfs = {"customer": customer_hist_clean, "product": product_hist_clean, "sales": sales_hist_clean}
raw_incr_counts = {
    "customer": df_customer.filter("source_load = 'incremental'").count(),
    "product":  df_product.filter("source_load = 'incremental'").count(),
    "sales":    df_sales.filter("source_load = 'incremental'").count(),
}
silver_audit_report = []
for ds in ["customer", "product", "sales"]:
    actual_hist = hist_clean_dfs[ds].count()
    expected_hist = int(spark.read.csv(f"{base_path}/audit_silver/{audit_silver_hist_files[ds]}", header=True, inferSchema=False).collect()[0]["row_count"])
    silver_audit_report.append({"table_name": f"{ds}_historical", "expected_row_count": expected_hist, "actual_row_count": actual_hist, "status": "PASS" if actual_hist == expected_hist else "FAIL"})
    actual_new = raw_incr_counts[ds]
    expected_new = int(spark.read.csv(f"{base_path}/audit_silver/{audit_silver_new_files[ds]}", header=True, inferSchema=False).collect()[0]["row_count"])
    silver_audit_report.append({"table_name": f"{ds}_new", "expected_row_count": expected_new, "actual_row_count": actual_new, "status": "PASS" if actual_new == expected_new else "FAIL"})
report_df = spark.createDataFrame(silver_audit_report)
display(report_df)
if any(r["status"] == "FAIL" for r in silver_audit_report):
    raise Exception("Silver audit validation FAILED")
print("✅ Silver audit checkpoints passed")

# ---------- PASS 1: Historical seeds Silver ----------
(customer_hist_clean.limit(0)
    .withColumn("effective_start_date", F.current_date()).withColumn("effective_end_date", F.lit(None).cast("date"))
    .withColumn("is_active", F.lit(True)).withColumn("customer_sk", F.lit(0).cast("long"))
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.silver_customer"))
(product_hist_clean.limit(0).withColumn("product_sk", F.lit(0).cast("long"))
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.silver_product"))
(sales_hist_clean.limit(0).withColumn("sales_sk", F.lit(0).cast("long"))
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.silver_sales"))

df_customer_hist_final = (customer_hist_clean
    .withColumn("customer_sk", F.row_number().over(Window.orderBy("customer_id")).cast("long"))
    .withColumn("effective_start_date", F.current_date()).withColumn("effective_end_date", F.lit(None).cast("date"))
    .withColumn("is_active", F.lit(True)))
df_customer_hist_final.write.format("delta").mode("append").saveAsTable(f"{catalog}.{schema}.silver_customer")

df_product_hist_final = product_hist_clean.withColumn("product_sk", F.row_number().over(Window.orderBy("product_id")).cast("long"))
df_product_hist_final.write.format("delta").mode("append").saveAsTable(f"{catalog}.{schema}.silver_product")

df_sales_hist_final = sales_hist_clean.withColumn("sales_sk", F.row_number().over(Window.orderBy("transaction_id")).cast("long"))
df_sales_hist_final.write.format("delta").mode("append").saveAsTable(f"{catalog}.{schema}.silver_sales")

print("PASS 1 done:", spark.table(f"{catalog}.{schema}.silver_customer").count(),
      spark.table(f"{catalog}.{schema}.silver_product").count(), spark.table(f"{catalog}.{schema}.silver_sales").count())

# ---------- PASS 2: Customers SCD2 ----------
silver_customer_table = f"{catalog}.{schema}.silver_customer"
delta_customer = DeltaTable.forName(spark, silver_customer_table)
tracked_cols = ["age", "gender", "income_bracket", "loyalty_program", "membership_years", "churned",
                "marital_status", "number_of_children", "education_level", "occupation",
                "customer_zip_code", "customer_city", "customer_state"]

existing_active = spark.table(silver_customer_table).filter("is_active = true")
existing_active_ids = set(row.customer_id for row in existing_active.select("customer_id").collect())

change_condition = " OR ".join([f"src.{c} <> tgt.{c}" for c in tracked_cols])
changed_df = (customer_incr_clean.alias("src").join(existing_active.alias("tgt"), "customer_id").where(change_condition).select("src.customer_id"))
changed_ids = [row.customer_id for row in changed_df.collect()]

if changed_ids:
    delta_customer.update(condition=f"customer_id IN ({','.join([repr(i) for i in changed_ids])}) AND is_active = true",
                           set={"is_active": "false", "effective_end_date": "current_date()"})

incr_ids = set(row.customer_id for row in customer_incr_clean.select("customer_id").collect())
new_customer_ids = incr_ids - existing_active_ids
to_insert_ids = set(changed_ids) | new_customer_ids
df_to_insert = customer_incr_clean.filter(F.col("customer_id").isin(list(to_insert_ids)))

max_sk = spark.table(silver_customer_table).agg(F.max("customer_sk")).collect()[0][0] or 0
df_to_insert_final = (df_to_insert
    .withColumn("customer_sk", (F.row_number().over(Window.orderBy("customer_id")) + F.lit(max_sk)).cast("long"))
    .withColumn("effective_start_date", F.current_date()).withColumn("effective_end_date", F.lit(None).cast("date"))
    .withColumn("is_active", F.lit(True)))
df_to_insert_final.write.format("delta").mode("append").saveAsTable(silver_customer_table)

print(f"SCD2 customers: {len(changed_ids)} expired, {len(new_customer_ids)} brand-new, {df_to_insert_final.count()} inserted")
print("silver_customer total:", spark.table(silver_customer_table).count(), "| active:", spark.table(silver_customer_table).filter("is_active=true").count())

# ---------- PASS 2: Products SCD1 ----------
silver_product_table = f"{catalog}.{schema}.silver_product"
delta_product = DeltaTable.forName(spark, silver_product_table)
max_sk_p = spark.table(silver_product_table).agg(F.max("product_sk")).collect()[0][0] or 0
df_product_incr_sk = product_incr_clean.withColumn("product_sk", (F.row_number().over(Window.orderBy("product_id")) + F.lit(max_sk_p)).cast("long"))
(delta_product.alias("t").merge(df_product_incr_sk.alias("s"), "t.product_id = s.product_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
print("silver_product total:", spark.table(silver_product_table).count())

# ---------- PASS 2: Sales Ledger ----------
silver_sales_table = f"{catalog}.{schema}.silver_sales"
delta_sales = DeltaTable.forName(spark, silver_sales_table)
w_s = Window.partitionBy("transaction_id").orderBy(F.col("ingested_at").desc())
df_sales_incr_dedup = sales_incr_clean.withColumn("rn", F.row_number().over(w_s)).filter("rn = 1").drop("rn")
max_sk_s = spark.table(silver_sales_table).agg(F.max("sales_sk")).collect()[0][0] or 0
df_sales_incr_sk = df_sales_incr_dedup.withColumn("sales_sk", (F.row_number().over(Window.orderBy("transaction_id")) + F.lit(max_sk_s)).cast("long"))
(delta_sales.alias("t").merge(df_sales_incr_sk.alias("s"), "t.transaction_id = s.transaction_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
print("silver_sales total:", spark.table(silver_sales_table).count())

print("=== SILVER LAYER COMPLETE ===")

customer historical: 1052 raw -> 1050 cleaned
customer incremental: 1053 raw -> 1050 cleaned
product historical: 1043 raw -> 1041 cleaned
product incremental: 1041 raw -> 1041 cleaned
sales historical: 1002 raw -> 1000 cleaned
sales incremental: 1000 raw -> 1000 cleaned


actual_row_count,expected_row_count,status,table_name
1050,1050,PASS,customer_historical
1053,1053,PASS,customer_new
1041,1041,PASS,product_historical
1041,1041,PASS,product_new
1000,1000,PASS,sales_historical
1000,1000,PASS,sales_new


✅ Silver audit checkpoints passed


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


PASS 1 done: 1050 1041 1000


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


SCD2 customers: 6 expired, 0 brand-new, 6 inserted
silver_customer total: 1056 | active: 1050
silver_product total: 1041


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


silver_sales total: 2000
=== SILVER LAYER COMPLETE ===


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
